# alphaXiv MCP Discovery Test

This notebook tests direct paper discovery through alphaXiv MCP. It does not use Feyman, Semantic Scholar, or OpenAlex.

Flow:

```text
User query -> alphaXiv MCP discover_papers -> local Paper records -> arXiv PDF URLs -> optional PyMuPDF ingestion
```

## 1. Setup project imports

In [ ]:
import os

import os

api_key = os.getenv("GROQ_API_KEY")
os.environ["GROQ_MODEL"] = "llama-3.1-70b-versatile"

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ingestion.paper_discovery import AlphaXivMCPPaperDiscovery, _alphaxiv_argument_sets

QUERY = "earthquake analysis using deep learning"
MAX_RESULTS = 5
DIFFICULTY = 5

ALPHAXIV_MCP_COMMAND = "npx -y mcp-remote https://api.alphaxiv.org/mcp/v1"

# See the planned alphaXiv calls first
plans = _alphaxiv_argument_sets(
    QUERY,
    DIFFICULTY,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    groq_model=os.getenv("GROQ_MODEL"),
)
plans

In [ ]:
discovery = AlphaXivMCPPaperDiscovery(
    command=ALPHAXIV_MCP_COMMAND,
    difficulty=DIFFICULTY,
    timeout=120,
    use_groq_planner=True,
)

papers = discovery.search(query=QUERY, max_results=MAX_RESULTS)

print(f"Found {len(papers)} papers")
for i, paper in enumerate(papers, start=1):
    print("=" * 80)
    print(f"{i}. {paper.title}")
    print("Paper ID:", paper.paper_id)
    print("Authors:", ", ".join(paper.authors[:5]))
    print("Published:", paper.published)
    print("PDF URL:", paper.pdf_url)
    print("Entry URL:", paper.entry_id)
    print("Abstract:", paper.summary[:500], "...")